In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# ==============================================================================
# --- 1. 配置 (这是您需要修改的核心部分) ---
# ==============================================================================

# Hydra 配置名
SAM2_CFG_NAME = "configs/sam2.1/sam2.1_hiera_b+"

# !! 关键 !! 指向【官方】的、未经微调的SAM2权重
SAM2_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2/checkpoints/sam2.1_hiera_base_plus.pt"

# !! 关键 !! 指向您为SAM2格式化后的【Kvasir-Sessile】数据集
DATASET_ROOT = "/home/zhengsongming/jupyterworkspace/datasets/Kvasir-SEG_for_SAM2"

# 指定要评估哪个划分：'test.txt' 或 'val.txt'
# 为了和I-MedSAM论文对标，我们使用测试集
EVALUATION_SPLIT_FILE = os.path.join(DATASET_ROOT, "ImageSets","2017","val.txt")


# ==============================================================================
# --- 2. 初始化SAM2模型和预测器 (无需修改) ---
# ==============================================================================
print("Loading ORIGINAL SAM2 model...")
device = "cuda" if torch.cuda.is_available() else "cpu"

# 注意：这里不传递任何hydra_overrides，以确保加载的是原始模型结构
model = build_sam2(SAM2_CFG_NAME, SAM2_CHECKPOINT_PATH, device=device)
predictor = SAM2ImagePredictor(model)
print("Model and Predictor loaded.")


# ==============================================================================
# --- 3. 定义指标计算函数 (无需修改) ---
# ==============================================================================
def calculate_metrics(gt_mask, pred_mask):
    """计算Dice系数和IoU"""
    gt_mask_bool = gt_mask > 0
    pred_mask_bool = pred_mask > 0
    intersection = np.logical_and(gt_mask_bool, pred_mask_bool).sum()
    dice = (2. * intersection) / (gt_mask_bool.sum() + pred_mask_bool.sum() + 1e-8)
    union = gt_mask_bool.sum() + pred_mask_bool.sum() - intersection
    iou = intersection / (union + 1e-8)
    return dice, iou


# ==============================================================================
# --- 4. 遍历数据集、执行分割并评估 (已修改为读取 .txt 文件) ---
# ==============================================================================
# 检查文件是否存在
if not os.path.exists(EVALUATION_SPLIT_FILE):
    raise FileNotFoundError(f"评估集文件未找到: {EVALUATION_SPLIT_FILE}")

# 读取样本名称
with open(EVALUATION_SPLIT_FILE, 'r') as f:
    sample_names = [line.strip() for line in f.readlines()]

print(f"\nFound {len(sample_names)} samples in the evaluation set. Starting...")

results = []
images_dir = os.path.join(DATASET_ROOT, "JPEGImages")
annotations_dir = os.path.join(DATASET_ROOT, "Annotations")

for sample_name in tqdm(sample_names, desc="Evaluating"):
    # 构建文件路径
    image_path = os.path.join(images_dir, sample_name, "00000.jpg")
    mask_path = os.path.join(annotations_dir, sample_name, "00000.png")

    if not os.path.exists(image_path) or not os.path.exists(mask_path):
        print(f"Warning: Files for sample '{sample_name}' not found. Skipping.")
        continue

    # 读取图像和掩码
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # --- 分割流程 (与您原来的代码完全相同) ---
    predictor.set_image(image_rgb)
    contours, _ = cv2.findContours(gt_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        print(f"Warning: No lesion found in mask for {sample_name}. Skipping.")
        continue
    
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    box_prompt = np.array([[x, y, x + w, y + h]])

    masks, _, _ = predictor.predict(box=box_prompt, multimask_output=False)
    pred_mask = (masks[0] * 255).astype(np.uint8)
    
    # --- 评估 ---
    dice, iou = calculate_metrics(gt_mask, pred_mask)
    
    # 记录结果 (这里只有一个类别，'polyp')
    results.append({"dice": dice, "iou": iou})
    
    predictor.reset_predictor()


# ==============================================================================
# --- 5. 打印最终的总结报告 (已简化) ---
# ==============================================================================
if results:
    df = pd.DataFrame(results)
    
    # 计算总体平均值和标准差
    mean_dice = df['dice'].mean()
    std_dice = df['dice'].std()
    mean_iou = df['iou'].mean()
    std_iou = df['iou'].std()
    count = len(df)
    
    print("\n" + "="*50)
    print("--- SAM2 Zero-Shot Evaluation on Kvasir-Sessile ---")
    print("="*50)
    print(f"Evaluated on: {os.path.basename(EVALUATION_SPLIT_FILE)}")
    print(f"Processed {count} images.")
    print(f"\nMean Dice: {mean_dice:.4f}  (Std: {std_dice:.4f})")
    print(f"Mean IoU : {mean_iou:.4f}  (Std: {std_iou:.4f})")
    print("="*50)
else:
    print("No images were processed. Please check the dataset path and structure.")

print("\n--- Evaluation complete! ---")

Loading ORIGINAL SAM2 model...
Model and Predictor loaded.

Found 200 samples in the evaluation set. Starting...


Evaluating: 100%|██████████| 200/200 [00:21<00:00,  9.52it/s]



--- SAM2 Zero-Shot Evaluation on Kvasir-Sessile ---
Evaluated on: val.txt
Processed 200 images.

Mean Dice: 0.9236  (Std: 0.0646)
Mean IoU : 0.8638  (Std: 0.0952)

--- Evaluation complete! ---
